<a href="https://colab.research.google.com/github/moridin04/DisasterResponseAssistant/blob/main/DisasterPreparednessAssistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Disaster Preparedness Assistant


- Upload the dataset CSV
- Preprocesses hazard columns
- Training a 'RandomForest classifier'
- IsolationForest for flagging anomalies
- Saving trained model(s) and processed data for backend use

In [ ]:
#Imports
import warnings
warnings.filterwarnings('ignore')

import io
import numpy as np
import pandas as pd
from google.colab import files

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, IsolationForest
import joblib

In [ ]:
#Uploading CSV dataset
print('Upload CSV file')
uploaded = files.upload()
if len(uploaded) == 0:
    raise RuntimeError('No file uploaded')

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print('DF shape:', df.shape)
df.head()

In [ ]:
#Preprocessing
risk_map = {'low': 1, 'medium': 2, 'high': 3}
non_hazard_cols = ['NCR', 'place', 'cluster', 'predicted_risk', 'lat', 'lon', 'recommendation']

hazard_cols = [c for c in df.columns if c not in non_hazard_cols]
print('Detected hazard columns:', hazard_cols)
df_proc = df.copy()
for c in hazard_cols:
    if df_proc[c].dtype == object or df_proc[c].dtype.name == 'category':
        df_proc[c] = df_proc[c].astype(str).str.strip().str.lower().map(risk_map)

imp = SimpleImputer(strategy='most_frequent')
df_proc[hazard_cols] = imp.fit_transform(df_proc[hazard_cols])

scaler = StandardScaler()
X = df_proc[hazard_cols].astype(float).values
print('Feature matrix X shape:', X.shape)

In [ ]:
#Supervised Training: RandomForest
rf_model = None
label_encoder = None
if 'predicted_risk' in df_proc.columns and df_proc['predicted_risk'].notna().any():
    print('predicted_risk found — preparing labels')

    if df_proc['predicted_risk'].dtype == object or df_proc['predicted_risk'].dtype.name == 'category':
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(df_proc['predicted_risk'].astype(str))
        print('Label classes:', list(label_encoder.classes_))
    else:
        y = df_proc['predicted_risk'].astype(int).values

    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except Exception:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    print('\nRandomForest classification report:')
    print(classification_report(y_test, y_pred, zero_division=0))
    print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

    joblib.dump({'model': rf_model, 'hazard_cols': hazard_cols, 'imputer': imp, 'scaler': scaler, 'label_encoder': label_encoder}, 'rf_model.joblib')
    print('\nRandomForest model saved to rf_model.joblib')
else:
    print("No 'predicted_risk' labels found — skipping RandomForest training. If you want RF training, include a 'predicted_risk' column in the uploaded CSV.")

In [ ]:
#Anomaly detection (IsolationForest)
iso = IsolationForest(contamination=0.02, random_state=42)
iso.fit(X)
outlier_flags = iso.predict(X)  # -1 -> outlier, 1 -> normal
df_proc['anomaly'] = (outlier_flags == -1)
print('Anomaly count:', int(df_proc['anomaly'].sum()))

joblib.dump({'model': iso, 'hazard_cols': hazard_cols, 'imputer': imp, 'scaler': scaler}, 'iso_model.joblib')
print('IsolationForest model saved to iso_model.joblib')

processed_csv = 'processed_hazards_with_anomalies.csv'
df_proc.to_csv(processed_csv, index=False)
print('Processed data saved to', processed_csv)

In [ ]:
#Load RF/ISO models and run prediction on new row
def load_rf_and_predict(sample_row_dict):
    import os
    if not os.path.exists('rf_model.joblib'):
        raise FileNotFoundError('rf_model.joblib not found. Train and save the model first.')
    pack = joblib.load('rf_model.joblib')
    model = pack['model']
    cols = pack['hazard_cols']
    imputer = pack['imputer']
    scaler = pack['scaler']
    le = pack.get('label_encoder', None)

    # Build sample df
    sample = {c: sample_row_dict.get(c, np.nan) for c in cols}
    sample_df = pd.DataFrame([sample])
    # map textual labels if present
    for c in cols:
        if sample_df[c].dtype == object:
            sample_df[c] = sample_df[c].astype(str).str.strip().str.lower().map({'low':1,'medium':2,'high':3})
    sample_df[cols] = imputer.transform(sample_df[cols])
    Xs = scaler.transform(sample_df[cols].astype(float).values)
    pred = model.predict(Xs)
    if le is not None:
        return le.inverse_transform(pred)[0]
    else:
        return int(pred[0])

print('Helper function defined: load_rf_and_predict(sample_row_dict)')